# Vídeo para YouTube desde Colab

Abrir: https://colab.research.google.com/github/DreiSup/spanishnav.al-podcast/blob/claude/quirky-dirac-x18c3n/video/video.ipynb

Hace el `.mp4` de un episodio sin pasar por tu ordenador: **portada fija + master**, y lo deja en Drive, en la carpeta
`video/` del episodio. Es el mismo vídeo que el comando de `ffmpeg` del SOP (`docs/sop-publicacion.md`, apartado 5 bis).

**Entorno: CPU.** No hace falta GPU: *Entorno de ejecución → Cambiar tipo de entorno de ejecución → CPU*.

Uso: pon el episodio en la celda 1 y ejecuta todas las celdas (*Entorno de ejecución → Ejecutar todas*). Drive pedirá
permiso la primera vez.

Dos modos:

- **`rapido`** (por defecto): codifica una vez un clip de 60 s con la portada y lo repite durante todo el audio sin
  volver a codificarlo. Lo que más tarda es pasar el audio a AAC: unos 20–35 s por cada 7 min de audio
  (probado el 5-nov fuera de Colab; un episodio de 3 h, unos 10–15 min).
- **`normal`**: el comando del SOP tal cual (codifica el vídeo entero: unas 5 veces más lento). Para comparar, o si el rápido diera problemas.

In [ ]:
# 1. Qué episodio (la carpeta de episodios/ del repo)
EPISODIO = "2019/002-ethical-wealth"
MODO = "rapido"        # "rapido" o "normal"
SOBRESCRIBIR = False   # True = rehace el vídeo aunque ya exista en Drive

In [ ]:
# 2. Drive, ffmpeg y la portada
import os, subprocess, shutil, time, json
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)
print(subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0])

PORTADA = Path("/content/portada-youtube.jpg")
subprocess.run(["curl", "-sSfL", "-o", str(PORTADA), "https://spanishnav.al/portada-youtube.jpg"], check=True)
print("Portada:", PORTADA.stat().st_size, "bytes")

In [ ]:
# 3. Buscar el master en Drive: naval-podcast/transcripts/<año>/<NNN Título>/audio/<NNN>-<slug>.mp3
RAIZ = Path("/content/drive/MyDrive/naval-podcast/transcripts")
anio, nombre = EPISODIO.split("/")
numero = nombre.split("-")[0]
if numero.isdigit():
    candidatas = sorted(p for p in (RAIZ / anio).glob(f"{numero} *") if p.is_dir())
    assert len(candidatas) == 1, f"Esperaba una carpeta '{numero} …' en {RAIZ / anio}, hay {len(candidatas)}: {candidatas}"
    CARPETA = candidatas[0]
else:
    CARPETA = RAIZ / anio / nombre
MASTER = CARPETA / "audio" / f"{nombre}.mp3"
assert MASTER.is_file(), f"No está el master: {MASTER}"

def duracion(ruta, flujo=None):
    cmd = ["ffprobe", "-v", "error"]
    if flujo:
        cmd += ["-select_streams", flujo, "-show_entries", "stream=duration"]
    else:
        cmd += ["-show_entries", "format=duration"]
    salida = subprocess.run(cmd + ["-of", "csv=p=0", str(ruta)], capture_output=True, text=True, check=True).stdout
    return float(salida.strip().splitlines()[0])

DUR = duracion(MASTER)
DESTINO = CARPETA / "video" / f"{nombre}.mp4"
print("Carpeta:", CARPETA)
print(f"Master:  {MASTER.name}  ({DUR / 60:.1f} min)")
print("Destino:", DESTINO, "(ya existe)" if DESTINO.exists() else "")

In [ ]:
# 4. Hacer el vídeo (en el disco de Colab, que es más rápido que Drive)
assert SOBRESCRIBIR or not DESTINO.exists(), "El vídeo ya existe en Drive. Pon SOBRESCRIBIR = True para rehacerlo."
VF = "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2,format=yuv420p"
SALIDA = Path("/content") / f"{nombre}.mp4"
AUDIO = ["-c:a", "aac", "-b:a", "192k", "-ar", "48000"]

def ffmpeg(*args):
    subprocess.run(["ffmpeg", "-v", "error", "-y", *args], check=True)

t0 = time.time()
if MODO == "rapido":
    CLIP = Path("/content/clip.mp4")
    ffmpeg("-loop", "1", "-framerate", "25", "-i", str(PORTADA), "-t", "60", "-vf", VF,
           "-c:v", "libx264", "-preset", "veryfast", "-tune", "stillimage", "-crf", "20", "-an", str(CLIP))
    # El clip se repite sin volver a codificarlo. La duración se fija con -t (la del master):
    # con -stream_loop -1, -shortest no corta y el fichero crecería sin fin.
    ffmpeg("-stream_loop", "-1", "-i", str(CLIP), "-i", str(MASTER), "-map", "0:v:0", "-map", "1:a:0",
           "-t", f"{DUR:.3f}", "-c:v", "copy", *AUDIO, "-movflags", "+faststart", str(SALIDA))
elif MODO == "normal":
    ffmpeg("-loop", "1", "-framerate", "25", "-i", str(PORTADA), "-i", str(MASTER), "-vf", VF,
           "-c:v", "libx264", "-preset", "veryfast", "-tune", "stillimage", "-crf", "20", *AUDIO,
           "-shortest", "-movflags", "+faststart", str(SALIDA))
else:
    raise ValueError(f"MODO desconocido: {MODO}")
print(f"Hecho en {time.time() - t0:.0f} s")

In [ ]:
# 5. Comprobar: vídeo y audio duran lo mismo que el master (±1 s)
dv, da = duracion(SALIDA, "v:0"), duracion(SALIDA, "a:0")
print(f"master {DUR:.1f} s · vídeo {dv:.1f} s · audio {da:.1f} s · {SALIDA.stat().st_size / 1e6:.1f} MB")
assert abs(dv - DUR) <= 1 and abs(da - DUR) <= 1, "Las duraciones no cuadran: no lo subas a YouTube"
print("OK")

In [ ]:
# 6. Copiar a Drive (carpeta video/ del episodio)
DESTINO.parent.mkdir(exist_ok=True)
shutil.copyfile(SALIDA, DESTINO)
print("En Drive:", DESTINO)
print("Siguiente: pide a la sesión de dirección el título y la descripción, y sube el vídeo en YouTube Studio.")